In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

In [22]:
DATA_PATH = "Data/Walmart.csv"
STRONG_WEEK_FACTOR    = 1.10
COLD_TEMP_F           = 40
EXPENSIVE_FUEL_USD    = 3.50
HIGH_UNEMPLOYMENT_PCT = 8
LARGE_STORE_USD       = 1_000_000
HOLIDAY_SEASON_MONTHS = [11, 12]
EVENT_NAMES = ["S", "H", "Q", "C", "F", "J", "G"]

## Event definitions

| Event | Definition |
|---|---|
| S | Strong week: Weekly_Sales ≥ 1.10 × that store's average weekly sales |
| H | Holiday week: Holiday_Flag = 1 |
| Q | Holiday season: week is in November or December |
| C | Cold week: Temperature < 40 °F |
| F | Expensive fuel: Fuel_Price ≥ $3.50 |
| J | High unemployment: Unemployment ≥ 8 % |
| G | Large store: the store's average weekly sales ≥ $1,000,000 |

In [23]:
sales_df = pd.read_csv(DATA_PATH)
sales_df["Date"] = pd.to_datetime(sales_df["Date"], dayfirst=True)
n_rows = len(sales_df)
n_stores = sales_df["Store"].nunique()
n_weeks = sales_df["Date"].nunique()
n_holiday_store_weeks = int(sales_df["Holiday_Flag"].sum())
n_missing = int(sales_df.isna().sum().sum())
print(f"Rows: {n_rows}")
print(f"Stores: {n_stores}")
print(f"Weeks: {n_weeks}")
print(f"Holiday store-weeks: {n_holiday_store_weeks}")
print(f"Missing values: {n_missing}")
print("\nAll data checks passed.")

Rows: 6435
Stores: 45
Weeks: 143
Holiday store-weeks: 450
Missing values: 0

All data checks passed.


In [4]:
def add_events(df, store_avg_sales):
    events_df = df.copy()
    avg_sales = events_df["Store"].map(store_avg_sales)
    events_df["S"] = events_df["Weekly_Sales"] >= STRONG_WEEK_FACTOR * avg_sales
    events_df["H"] = events_df["Holiday_Flag"] == 1
    events_df["Q"] = events_df["Date"].dt.month.isin(HOLIDAY_SEASON_MONTHS)
    events_df["C"] = events_df["Temperature"] < COLD_TEMP_F
    events_df["F"] = events_df["Fuel_Price"] >= EXPENSIVE_FUEL_USD
    events_df["J"] = events_df["Unemployment"] >= HIGH_UNEMPLOYMENT_PCT
    events_df["G"] = avg_sales >= LARGE_STORE_USD
    return events_df

store_avg_all = sales_df.groupby("Store")["Weekly_Sales"].mean()
events_all = add_events(sales_df, store_avg_all)

event_rows = []
for event in EVENT_NAMES:
    count_event = int(events_all[event].sum())
    count_complement = int((~events_all[event]).sum())
    p_event = count_event / n_rows
    p_complement = count_complement / n_rows
    event_rows.append({
        "event": event,
        "count": count_event,
        "P(E)": p_event,
        "P(E^c)": p_complement,
        "P(E) + P(E^c)": p_event + p_complement,   # R5 check: must be 1
    })

event_table = pd.DataFrame(event_rows).set_index("event")

print(f"P(H) = {n_holiday_store_weeks}/{n_rows} = {event_table.loc['H', 'P(E)']:.4f}")
event_table.round(4)

P(H) = 450/6435 = 0.0699


,count,P(E),P(E^c),P(E) + P(E^c)
event,,,,
S,814,0.1265,0.8735,1.0
H,450,0.0699,0.9301,1.0
Q,810,0.1259,0.8741,1.0
C,989,0.1537,0.8463,1.0
F,3029,0.4707,0.5293,1.0
J,3022,0.4696,0.5304,1.0
G,3146,0.4889,0.5111,1.0


In [5]:
joint_pairs = [("S", "H"), ("S", "Q"), ("H", "Q"), ("S", "C")]

joint_rows = []
for event_a, event_b in joint_pairs:
    a_and_b = events_all[event_a] & events_all[event_b]
    a_and_not_b = events_all[event_a] & ~events_all[event_b]
    p_a_and_b = a_and_b.mean()
    p_a_and_not_b = a_and_not_b.mean()
    joint_rows.append({
        "pair": f"{event_a} ∩ {event_b}",
        "count": int(a_and_b.sum()),
        "P(A ∩ B)": p_a_and_b,
        "P(A ∩ B^c)": p_a_and_not_b,
        "P(A) rebuilt": p_a_and_b + p_a_and_not_b,
        "P(A) direct": events_all[event_a].mean(),
    })

joint_table = pd.DataFrame(joint_rows).set_index("pair")
print("All P3 checks passed.")
joint_table.round(4)

All P3 checks passed.


,count,P(A ∩ B),P(A ∩ B^c),P(A) rebuilt,P(A) direct
pair,,,,,
S ∩ H,122,0.0190,0.1075,0.1265,0.1265
S ∩ Q,333,0.0517,0.0747,0.1265,0.1265
H ∩ Q,180,0.0280,0.0420,0.0699,0.0699
S ∩ C,185,0.0287,0.0977,0.1265,0.1265


In [6]:
p_S = events_all["S"].mean()
p_H = events_all["H"].mean()

# Conditional probability by counting inside each group
p_S_given_H = events_all.loc[events_all["H"], "S"].mean()
p_S_given_not_H = events_all.loc[~events_all["H"], "S"].mean()
holiday_lift = p_S_given_H / p_S_given_not_H

n_S_and_H     = int((events_all["S"] & events_all["H"]).sum())
n_S_and_not_H = int((events_all["S"] & ~events_all["H"]).sum())

print(f"P(S | H) = {n_S_and_H}/{int(events_all['H'].sum())} = {p_S_given_H:.4f}")
print(f"P(S | H^c) = {n_S_and_not_H}/{int((~events_all['H']).sum())} = {p_S_given_not_H:.4f}")
print(f"A strong week is {holiday_lift:.2f}x as likely in a holiday week.")
p_S_rebuilt_from_H = p_S_given_H * p_H + p_S_given_not_H * (1 - p_H)
print(f"Check: P(S|H)P(H) + P(S|H^c)P(H^c) = {p_S_rebuilt_from_H:.4f}  vs  P(S) = {p_S:.4f}")

P(S | H) = 122/450 = 0.2711
P(S | H^c) = 692/5985 = 0.1156
A strong week is 2.34x as likely in a holiday week.
Check: P(S|H)P(H) + P(S|H^c)P(H^c) = 0.1265  vs  P(S) = 0.1265


In a holiday week, a store has a strong week 27.1% of the time (122 of 450), compared with 11.6% in other weeks (692 of 5,985). A holiday week makes a strong week about 2.3 times as likely, but most holiday weeks (73%) are still not strong.

In [7]:
p_H_given_S_bayes = p_S_given_H * p_H / p_S                  # Bayes' theorem
p_H_given_S_count = events_all.loc[events_all["S"], "H"].mean()  # count inside strong weeks

print(f"P(H | S) by Bayes: {p_H_given_S_bayes}")
print(f"P(H | S) by counting: {p_H_given_S_count}")
print(f"Difference: {abs(p_H_given_S_bayes - p_H_given_S_count)}")

P(H | S) by Bayes: 0.1498771498771499
P(H | S) by counting: 0.14987714987714987
Difference: 2.7755575615628914e-17


In [8]:
HOLIDAY_BY_MONTH = {2: "Super Bowl", 9: "Labor Day", 11: "Thanksgiving", 12: "Christmas"}
GROUP_ORDER = ["Super Bowl", "Labor Day", "Thanksgiving", "Christmas", "No holiday"]

# Holiday weeks get their holiday's name (by month); all other weeks are "No holiday"
events_all["holiday_group"] = np.where(
    events_all["H"],
    events_all["Date"].dt.month.map(HOLIDAY_BY_MONTH),
    "No holiday",
)

group_table = (
    events_all.groupby("holiday_group")
    .agg(n_weeks=("S", "size"), n_strong=("S", "sum"))
    .reindex(GROUP_ORDER)
)
group_table["P(g)"] = group_table["n_weeks"] / n_rows
group_table["P(S | g)"] = group_table["n_strong"] / group_table["n_weeks"]
group_table["P(S | g) P(g)"] = group_table["P(S | g)"] * group_table["P(g)"]
group_table["P(g | S)"] = group_table["P(S | g) P(g)"] / p_S   # Bayes: share of strong weeks

p_S_rebuilt = group_table["P(S | g) P(g)"].sum()   # law of total probability

print(f"Sum of P(g): {group_table['P(g)'].sum():.4f}")   # R5: groups cover S exactly once
print(f"P(S) rebuilt: {p_S_rebuilt:.4f}")
print(f"P(S) direct: {p_S:.4f}")
print(f"Highest P(S | g): {group_table['P(S | g)'].idxmax()}")
print(f"Largest P(g | S): {group_table['P(g | S)'].idxmax()}")
group_table.round(4)

Sum of P(g): 1.0000
P(S) rebuilt: 0.1265
P(S) direct: 0.1265
Highest P(S | g): Thanksgiving
Largest P(g | S): No holiday


,n_weeks,n_strong,P(g),P(S | g),P(S | g) P(g),P(g | S)
holiday_group,,,,,,
Super Bowl,135,26,0.0210,0.1926,0.0040,0.0319
Labor Day,135,16,0.0210,0.1185,0.0025,0.0197
Thanksgiving,90,74,0.0140,0.8222,0.0115,0.0909
Christmas,90,6,0.0140,0.0667,0.0009,0.0074
No holiday,5985,692,0.9301,0.1156,0.1075,0.8501


In [9]:
n_S_and_Q = int((events_all["S"] & events_all["Q"]).sum())
p_S_given_Q = events_all.loc[events_all["Q"], "S"].mean()   # inside the Nov–Dec weeks
p_Q_given_S = events_all.loc[events_all["S"], "Q"].mean()   # inside the strong weeks

print(f"P(S | Q) = {n_S_and_Q}/{int(events_all['Q'].sum())} = {p_S_given_Q:.4f}")
print(f"P(Q | S) = {n_S_and_Q}/{int(events_all['S'].sum())} = {p_Q_given_S:.4f}")
p_Q = events_all["Q"].mean()
print(f"Check (Bayes): P(Q|S) P(S) / P(Q) = {p_Q_given_S * p_S / p_Q:.4f}  vs  P(S|Q) = {p_S_given_Q:.4f}")

P(S | Q) = 333/810 = 0.4111
P(Q | S) = 333/814 = 0.4091
Check (Bayes): P(Q|S) P(S) / P(Q) = 0.4111  vs  P(S|Q) = 0.4111


P(S | Q) = 0.411 is the chance that a Nov–Dec week is strong (333 of 810), and P(Q | S) = 0.409 is the share of strong weeks that fall in Nov–Dec (333 of 814). They answer different questions: both have the same 333 weeks on top but divide by a different group. Here they come out almost equal only because Q and S happen to be nearly the same size (810 vs 814); in general, P(S | Q) = P(Q | S) · P(S) / P(Q), and they differ whenever P(S) is not equal to P(Q).

In [10]:
def evaluate_rule(rule_event):
    """Precision and recall of the rule 'add staff when rule_event is true'."""
    p_rule_given_S = events_all.loc[events_all["S"], rule_event].mean()
    p_rule_given_not_S = events_all.loc[~events_all["S"], rule_event].mean()

    # Bayes, with the denominator from the law of total probability
    p_rule = p_rule_given_S * p_S + p_rule_given_not_S * (1 - p_S)
    precision_bayes = p_rule_given_S * p_S / p_rule

    precision_count = events_all.loc[events_all[rule_event], "S"].mean()   # R5 check
    return {
        "P(rule | S)": p_rule_given_S,
        "P(rule | S^c)": p_rule_given_not_S,
        "precision (Bayes)": precision_bayes,
        "precision (count)": precision_count,
        "recall": p_rule_given_S,
        "missed share of strong weeks": 1 - p_rule_given_S,
    }

rule_table = pd.DataFrame({
    "Holiday rule (H)": evaluate_rule("H"),
    "Nov-Dec rule (Q)": evaluate_rule("Q"),
}).T
rule_table.round(4)

,P(rule | S),P(rule | S^c),precision (Bayes),precision (count),recall,missed share of strong weeks
Holiday rule (H),0.1499,0.0584,0.2711,0.2711,0.1499,0.8501
Nov-Dec rule (Q),0.4091,0.0849,0.4111,0.4111,0.4091,0.5909


The holiday rule is right 27.1% of the time it adds staff (precision, same by Bayes and by counting), but it catches only 15.0% of strong weeks (recall), so it misses 85% of them. The Nov–Dec rule is better on both measures: precision 41.1% and recall 40.9%. It still misses 59% of strong weeks.

In [11]:
def precision_at_base_rate(p_rule_given_S, p_rule_given_not_S, base_rate):
    numerator = p_rule_given_S * base_rate
    return numerator / (numerator + p_rule_given_not_S * (1 - base_rate))

p_H_given_S = rule_table.loc["Holiday rule (H)", "P(rule | S)"]
p_H_given_not_S = rule_table.loc["Holiday rule (H)", "P(rule | S^c)"]

base_rate_table = pd.DataFrame({
    "scenario": ["Actual", "Strong weeks twice as common", "Strong weeks half as common"],
    "P(S)": [p_S, 2 * p_S, p_S / 2],
}).set_index("scenario")
base_rate_table["holiday rule precision"] = [
    precision_at_base_rate(p_H_given_S, p_H_given_not_S, base_rate)
    for base_rate in base_rate_table["P(S)"]
]
base_rate_table.round(4)

,P(S),holiday rule precision
scenario,,
Actual,0.1265,0.2711
Strong weeks twice as common,0.2530,0.4652
Strong weeks half as common,0.0632,0.1478


In [12]:
INFO_EVENTS = ["H", "Q", "C", "F", "J", "G"]

independence_rows = []
for event_b in INFO_EVENTS:
    p_joint   = (events_all["S"] & events_all[event_b]).mean()
    p_product = p_S * events_all[event_b].mean()
    independence_rows.append({
        "pair": f"(S, {event_b})",
        "P(S ∩ B)": p_joint,
        "P(S) P(B)": p_product,
        "lift": p_joint / p_product,
        "P(S | B)": events_all.loc[events_all[event_b], "S"].mean(),
        "check: P(S ∩ B) / P(B)": p_joint / events_all[event_b].mean(),
        "P(S | B^c)": events_all.loc[~events_all[event_b], "S"].mean(),
    })

independence_table = pd.DataFrame(independence_rows).set_index("pair")
independence_table.round(4)

,P(S ∩ B),P(S) P(B),lift,P(S | B),check: P(S ∩ B) / P(B),P(S | B^c)
pair,,,,,,
"(S, H)",0.0190,0.0088,2.1432,0.2711,0.2711,0.1156
"(S, Q)",0.0517,0.0159,3.2500,0.4111,0.4111,0.0855
"(S, C)",0.0287,0.0194,1.4788,0.1871,0.1871,0.1155
"(S, F)",0.0452,0.0595,0.7595,0.0961,0.0961,0.1536
"(S, J)",0.0640,0.0594,1.0778,0.1363,0.1363,0.1178
"(S, G)",0.0570,0.0618,0.9222,0.1167,0.1167,0.1359


No pair is exactly independent, but the strength differs a lot. The holiday season Q carries the most information (lift 3.25: P(S | Q) = 0.41 vs 0.09 otherwise), then holiday weeks H (2.14) and cold weeks C (1.48). Expensive fuel F is negatively associated (lift 0.76: strong weeks are less common when fuel is expensive). High unemployment J (1.08) and large store G (0.92) are close to independent of S, so they tell us little about strong weeks.

In [13]:
TRAIN_LAST_YEAR = 2011
TEST_YEAR = 2012
FEATURES = ["H", "Q", "C", "F", "J", "G"]
ALPHA = 1   # Laplace smoothing, used both by hand (N2/N3) and in scikit-learn (N5)

train_raw = sales_df[sales_df["Date"].dt.year <= TRAIN_LAST_YEAR]
test_raw  = sales_df[sales_df["Date"].dt.year == TEST_YEAR]

# Each store's average from the training years; test weeks stay unseen
store_avg_train = train_raw.groupby("Store")["Weekly_Sales"].mean()
train_df = add_events(train_raw, store_avg_train)
test_df  = add_events(test_raw, store_avg_train)

print(f"Training: {len(train_df)} store-weeks ({train_df['Date'].min().date()} to {train_df['Date'].max().date()})")
print(f"Test: {len(test_df)} store-weeks ({test_df['Date'].min().date()} to {test_df['Date'].max().date()})")
print(f"Total: {len(train_df) + len(test_df)} (should be {n_rows})")
print(f"P(S) training = {int(train_df['S'].sum())}/{len(train_df)} = {train_df['S'].mean():.4f}")
print(f"P(S) test = {int(test_df['S'].sum())}/{len(test_df)} = {test_df['S'].mean():.4f}")

Training: 4500 store-weeks (2010-02-05 to 2011-12-30)
Test: 1935 store-weeks (2012-01-06 to 2012-10-26)
Total: 6435 (should be 6435)
P(S) training = 574/4500 = 0.1276
P(S) test = 255/1935 = 0.1318


In [14]:
is_strong_train = train_df["S"]
n_strong_train = int(is_strong_train.sum())
n_not_strong_train = int((~is_strong_train).sum())

prior_S = n_strong_train / len(train_df)
prior_not_S = n_not_strong_train / len(train_df)
print(f"Priors: P(S) = {prior_S:.4f}, P(S^c) = {prior_not_S:.4f}, sum = {prior_S + prior_not_S:.4f}")

likelihood_rows = []
for feature in FEATURES:
    count_given_S = int(train_df.loc[is_strong_train, feature].sum())
    count_given_not_S = int(train_df.loc[~is_strong_train, feature].sum())
    likelihood_rows.append({
        "feature": feature,
        "count | S": count_given_S,
        "P(f=1 | S)": (count_given_S + ALPHA) / (n_strong_train + 2 * ALPHA),
        "count | S^c": count_given_not_S,
        "P(f=1 | S^c)": (count_given_not_S + ALPHA) / (n_not_strong_train + 2 * ALPHA),
    })

likelihood_table = pd.DataFrame(likelihood_rows).set_index("feature")
likelihood_table["P(f=0 | S)"] = 1 - likelihood_table["P(f=1 | S)"]     # complements (R5)
likelihood_table["P(f=0 | S^c)"] = 1 - likelihood_table["P(f=1 | S^c)"]
likelihood_table.round(4)

Priors: P(S) = 0.1276, P(S^c) = 0.8724, sum = 1.0000


,count | S,P(f=1 | S),count | S^c,P(f=1 | S^c),P(f=0 | S),P(f=0 | S^c)
feature,,,,,,
H,99,0.1736,261,0.0667,0.8264,0.9333
Q,335,0.5833,475,0.1212,0.4167,0.8788
C,165,0.2882,599,0.1527,0.7118,0.8473
F,108,0.1892,1358,0.3460,0.8108,0.6540
J,317,0.5521,2085,0.5311,0.4479,0.4689
G,266,0.4635,1934,0.4926,0.5365,0.5074


In [15]:
PROFILE_A = {"H": 1, "Q": 1, "C": 1, "F": 0, "J": 0, "G": 1}   # holiday, Nov–Dec, cold, cheap fuel, low unemp., large
PROFILE_B = {"H": 0, "Q": 0, "C": 0, "F": 1, "J": 1, "G": 0}   # none of those, expensive fuel, high unemp., small

def naive_bayes_by_hand(profile):
    """Return the factor table, both class scores, and the normalized P(S | x)."""
    score_S, score_not_S = prior_S, prior_not_S
    factor_rows = []
    for feature in FEATURES:
        value = profile[feature]
        factor_S = likelihood_table.loc[feature, f"P(f={value} | S)"]
        factor_not_S = likelihood_table.loc[feature, f"P(f={value} | S^c)"]
        score_S *= factor_S
        score_not_S *= factor_not_S
        factor_rows.append({"feature": feature, "value": value,
                            "P(f | S)": factor_S, "P(f | S^c)": factor_not_S})
    posterior_S = score_S / (score_S + score_not_S)   # normalization
    return pd.DataFrame(factor_rows).set_index("feature"), score_S, score_not_S, posterior_S

hand_posteriors = {}
for profile_name, profile in [("holiday week, Nov–Dec,cold, cheap fuel, low unemployment, large store", PROFILE_A), 
("no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store", PROFILE_B)]:
    factor_table, score_S, score_not_S, posterior_S = naive_bayes_by_hand(profile)
    hand_posteriors[profile_name] = posterior_S
    print(f"\n Profile: {profile_name}")
    display(factor_table.round(4))
    print(f"Score(S) = {prior_S:.4f} x product of P(f | S)   = {score_S:.6f}")
    print(f"Score(S^c) = {prior_not_S:.4f} x product of P(f | S^c) = {score_not_S:.6f}")
    print(f"P(S | x) = {score_S:.6f} / ({score_S:.6f} + {score_not_S:.6f}) = {posterior_S:.4f}")


 Profile: holiday week, Nov–Dec,cold, cheap fuel, low unemployment, large store


,value,P(f | S),P(f | S^c)
feature,,,
H,1,0.1736,0.0667
Q,1,0.5833,0.1212
C,1,0.2882,0.1527
F,0,0.8108,0.6540
J,0,0.4479,0.4689
G,1,0.4635,0.4926


Score(S) = 0.1276 x product of P(f | S)   = 0.000627
Score(S^c) = 0.8724 x product of P(f | S^c) = 0.000163
P(S | x) = 0.000627 / (0.000627 + 0.000163) = 0.7939

 Profile: no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store


,value,P(f | S),P(f | S^c)
feature,,,
H,0,0.8264,0.9333
Q,0,0.4167,0.8788
C,0,0.7118,0.8473
F,1,0.1892,0.3460
J,1,0.5521,0.5311
G,0,0.5365,0.5074


Score(S) = 0.1276 x product of P(f | S)   = 0.001752
Score(S^c) = 0.8724 x product of P(f | S^c) = 0.056519
P(S | x) = 0.001752 / (0.001752 + 0.056519) = 0.0301


In [16]:
strong_train = train_df[is_strong_train]

p_H_and_Q_given_S = (strong_train["H"] & strong_train["Q"]).mean()
p_H_given_S_train = strong_train["H"].mean()
p_Q_given_S_train = strong_train["Q"].mean()
product_given_S = p_H_given_S_train * p_Q_given_S_train

print(f"P(H ∩ Q | S) = {p_H_and_Q_given_S:.4f}")
print(f"P(H | S) x P(Q | S) = {p_H_given_S_train:.4f} x {p_Q_given_S_train:.4f} = {product_given_S:.4f}")
print(f"Ratio = {p_H_and_Q_given_S / product_given_S:.2f}")

P(H ∩ Q | S) = 0.1394
P(H | S) x P(Q | S) = 0.1725 x 0.5836 = 0.1007
Ratio = 1.38


Among strong training weeks, P(H ∩ Q | S) = 0.139, but the naive product is 0.173 × 0.584 = 0.101, so the true joint is 1.38 times the product. H and Q are not independent because two of the four holidays (Thanksgiving and Christmas) always fall in Nov–Dec: knowing a week is a holiday week raises the chance it is in Nov–Dec. Naive Bayes counts this overlapping evidence twice, which tends to push its probabilities too far toward strong week for Nov–Dec holiday weeks.

In [17]:
X_train = train_df[FEATURES].astype(int)
y_train = train_df["S"].astype(int)
X_test = test_df[FEATURES].astype(int)
y_test = test_df["S"].astype(int)

nb_model = CategoricalNB(alpha=ALPHA).fit(X_train, y_train)

profiles_df = pd.DataFrame([PROFILE_A, PROFILE_B], index=["holiday week, Nov–Dec,cold, cheap fuel, low unemployment, large store",
"no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store"])[FEATURES]
sklearn_posteriors = nb_model.predict_proba(profiles_df)[:, 1]

comparison_table = pd.DataFrame({
    "P(S | x) by hand": [hand_posteriors["holiday week, Nov–Dec,cold, cheap fuel, low unemployment, large store"], 
    hand_posteriors["no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store"]],
    "P(S | x) scikit-learn": sklearn_posteriors,
}, index=["holiday week, Nov–Dec,cold, cheap fuel, low unemployment, large store", 
"no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store"])
comparison_table.round(6)

,P(S | x) by hand,P(S | x) scikit-learn
"holiday week, Nov–Dec,cold, cheap fuel, low unemployment, large store",0.793851,0.793851
"no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store",0.030069,0.030069


In [18]:
def evaluate_predictions(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),   # 0 when nothing is flagged
            "recall": recall_score(y_true, y_pred, zero_division=0)}

nb_predictions = nb_model.predict(X_test)   # default rule: predict S when P(S | x) > 0.5
print("Naive Bayes confusion matrix (rows = actual 0/1, columns = predicted 0/1):")
print(confusion_matrix(y_test, nb_predictions, labels=[0, 1]))

results_table = pd.DataFrame({
    "Naive Bayes": evaluate_predictions(y_test, nb_predictions),
    "Never strong": evaluate_predictions(y_test, np.zeros(len(y_test), dtype=int)),
    "Strong if holiday": evaluate_predictions(y_test, test_df["H"].astype(int)),
}).T

print(f"\nTest weeks in Nov–Dec (Q = 1): {int(test_df['Q'].sum())}")
print(f"Holiday store-weeks in test: {int(test_df['H'].sum())}  (Super Bowl and Labor Day only)")
results_table.round(4)

Naive Bayes confusion matrix (rows = actual 0/1, columns = predicted 0/1):
[[1680    0]
 [ 255    0]]

Test weeks in Nov–Dec (Q = 1): 0
Holiday store-weeks in test: 90  (Super Bowl and Labor Day only)


,TP,FP,FN,TN,accuracy,precision,recall
Naive Bayes,0.0,0.0,255.0,1680.0,0.8682,0.0000,0.0000
Never strong,0.0,0.0,255.0,1680.0,0.8682,0.0000,0.0000
Strong if holiday,23.0,67.0,232.0,1613.0,0.8455,0.2556,0.0902


At the default 0.5 threshold, Naive Bayes never predicts a strong week in 2012, so it is identical to the "never strong" baseline: accuracy 0.868, precision 0 and recall 0. The holiday rule has lower accuracy (0.845) but finds 23 of the 255 strong weeks (recall 0.090, precision 0.256), so it is the only one of the three that is useful for staffing. The reason is the test year: 2012 ends in October, so Q is 0 in every test week and the two holidays with the strongest effect (Thanksgiving and Christmas) never appear. Without Q, no test profile reaches P(S | x) > 0.5. Accuracy is a misleading measure here, because always saying "not strong" is already right 87% of the time.

In [19]:
THRESHOLDS = [0.1, 0.2, 0.3, 0.5]
test_posteriors = nb_model.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in THRESHOLDS:
    threshold_predictions = (test_posteriors > threshold).astype(int)
    threshold_rows.append({"threshold": threshold,
                           "weeks flagged": int(threshold_predictions.sum()),
                           **evaluate_predictions(y_test, threshold_predictions)})

threshold_table = pd.DataFrame(threshold_rows).set_index("threshold")
threshold_table.round(4)

,weeks flagged,TP,FP,FN,TN,accuracy,precision,recall
threshold,,,,,,,,
0.1,125,19,106,236,1574,0.8233,0.152,0.0745
0.2,10,3,7,252,1673,0.8661,0.300,0.0118
0.3,2,0,2,255,1678,0.8672,0.000,0.0000
0.5,0,0,0,255,1680,0.8682,0.000,0.0000


Lowering the threshold makes the model flag more weeks. At t = 0.1 it flags 125 store-weeks with precision 0.152 and recall 0.075; at t = 0.2 it flags only 10 (precision 0.30, recall 0.012); at t = 0.3 and 0.5 it flags almost none. Even at its best, the model does not beat the simple holiday rule on the 2012 test year (precision 0.256, recall 0.090).

### Note to Maria
Across the 45 stores, a store has a strong week (sales at least 10% above its own average) in 12.6% of weeks, or about 1 week in 8. Thanksgiving week matters most, with a strong week 82% of the time, and November–December as a whole raises the chance to 41%, compared with 9% in other months, while cold weeks help only a little (19%). The current holiday rule is right only 27% of the time it adds staff and catches just 15% of strong weeks, so it misses 85% of them; adding staff for all of November–December would be right 41% of the time and catch 41% of strong weeks. Our simple model did not beat the holiday rule: when tested on 2012, it predicted no strong weeks at all, while the holiday rule found 9% of them. The main risk is that the 2012 data ends in October, so the model was never tested on November–December, its strongest signal, and its results may look better or worse once a full holiday season is included.